# MSD replication with maximum workers — 3 folds × 3 seeds, no augmentation

**504 total runs, preserving completed and interrupted work:** four models × 14 conditions × folds 0/1/2 × seeds 0/1/2.
Each fold retains its saved 5% training subset (10 volumes) and 52 validation volumes.
The completed cluster pilot is provenance, not silently counted as a Colab replication.

- **No augmentation:** no random flips, rotations, intensity jitter, or test-time augmentation.
  The existing deterministic normalization/padding stays unchanged.
- nnUNet: native foreground Dice+CE, original deep supervision and SGD; 60% auxiliary targets, 75% p95 ceiling.
- SwinUNETR, UNETR and 3D U-Net: established Dice including background and AdamW; 10% targets, 50% p95 ceiling.
- Preserve BF16 precision, batch size 1, initialization, LR schedule, calibration, SDF timing and the
  31,976,450,667,090 optimization-FLOP cap per run. Calibration and preflight compute are separate.

**Before connecting a GPU:** upload `msd_data_verified.zip` and `msd_replication_code.zip` to
`My Drive/hippo_msd_3fold_3seed/input/`. Keep both zipped. No SSH keys or cluster access are needed.
Then choose **Runtime → Change runtime type → GPU**. Use a native-BF16 GPU such as H100,
A100, L4 or RTX PRO 6000 Blackwell;
a T4 cannot preserve this recipe. One notebook runtime at a time should use this results folder.

Run cells in order. All results/recovery go directly to Drive; your Mac is not an archive server.
Colab availability, session lifetime and background execution depend on your plan. Closing a laptop
is not a guarantee of uninterrupted Colab execution. See the [official FAQ](https://research.google.com/colaboratory/faq.html).


**Switching from the existing notebook:** interrupt its run cell once and wait until checkpoint
uploads finish. Keep the same A100 type, environment and results directory. Upload only
`msd_max_workers_control.zip` beside the two original ZIPs; open this notebook and run its cells in order.
The original science bundle is unchanged. Verified results are skipped, recovery has launch priority,
and an unchanged preflight is reused. Reuse the current runtime when possible. If Colab assigns a
fresh A100 runtime, verified recovery is restored from Drive. Never delete the old runtime before
its clean stop and recovery uploads finish; do not leave two sweeps running against these results.

**Keep worker slots filled:** the scheduler immediately targets the CPU-supported maximum and
fills slots whenever eligible work and memory/storage headroom are available. There are no solo
memory-profiling runs, gradual ramp-up or throughput-triggered reductions. Memory is observed
during ordinary concurrent runs. Scientific loss calibration still runs exactly as prescribed.
Concurrent execution is locally tested; its actual Colab throughput must be measured.

**Storage update:** when `/mnt/local-scratch` is a separate mounted disk, new run checkpoints
and recovery staging use it automatically. Existing run directories stay exactly where they are;
interrupted work has launch priority. No dataset upload or migration of active files is needed.
Google Drive archives, per-update charge writes, checkpoint format and cadence stay unchanged.

**Faster archive verification:** each saved archive file is read from Drive once per check.
Hashes, readability, identity and cleanup proofs are then checked locally with the original verifier.
Results and memory profiles are parsed from that same verified copy, which is discarded afterward.
Verification starts fresh on every restart; no old cache is trusted. Folder counts and the current
filename remain visible while Drive is slow. The results cell uses the same verification method.


## 1. Mount Drive and stage the unchanged science bundle plus scheduler


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import fcntl, hashlib, json, os, shutil, signal, subprocess, sys, zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive/hippo_msd_3fold_3seed')
INPUT = DRIVE_ROOT/'input'
RESULTS = DRIVE_ROOT/'results_noaug_v1'  # Preserve this exact existing results folder.
BUNDLE = Path('/content/hippo_msd_bundle')
CONTROL = Path('/content/hippo_msd_max_workers')
SCIENTIFIC_SHA = '3b9d0c30f71873b6f834ae3b1fd989bf33e37bef7682fdda094b656daff3620b'
EXPECTED = {'msd_data_verified.zip':'ff045aea37338769009e6322ed55fe08b654681e643ce9cb78782b80e11921f4', 'msd_replication_code.zip':'b0a6dc1a5b94ce8d7ae2506e1583401313cc60fae6735dd4c09f6c73f5e6c51b'}
CONTROL_SHA = 'aab553e34014bc4556f808c57e03c5932f600a8dba8f4147546654ad8287ed04'
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8*1024**2), b''):
            h.update(block)
    return h.hexdigest()

BUNDLE.mkdir(exist_ok=True)
(BUNDLE/'work').mkdir(exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
def unpack_verified(src, expected, destination):
    assert sha256(src) == expected, f'Upload hash mismatch: {src.name}'
    local = Path('/content')/src.name
    shutil.copyfile(src, local)
    assert sha256(local) == expected
    with zipfile.ZipFile(local) as archive:
        for info in archive.infolist():
            member = Path(info.filename)
            assert not member.is_absolute() and '..' not in member.parts
            assert not ((info.external_attr >> 16) & 0o170000) == 0o120000
        archive.extractall(destination)

with (BUNDLE/'work/sweep.lock').open('a') as lock:
    try:
        fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    except BlockingIOError:
        raise RuntimeError('The old sweep is running. Interrupt it once and wait for recovery before switching.')
    for proc in Path('/proc').glob('[0-9]*/cmdline'):
        try:
            words = proc.read_bytes().decode(errors='replace').split('\0')
        except (FileNotFoundError, PermissionError, ProcessLookupError):
            continue
        if str(BUNDLE/'entry.py') in words or (str(BUNDLE) in words and any(w.endswith('/worker.py') for w in words)):
            raise RuntimeError('An existing trainer is still running. Wait for its clean stop before setup.')
    manifest = BUNDLE/'bundle_manifest.json'
    if manifest.exists():
        assert sha256(manifest) == SCIENTIFIC_SHA, 'Existing bundle differs; stop and preserve it for inspection.'
        for name, expected in json.loads(manifest.read_text()).items():
            assert sha256(BUNDLE/name) == expected, f'Existing bundle file differs: {name}'
        print('Verified existing scientific bundle in place; work/checkpoints preserved.')
    else:
        assert not any((BUNDLE/'work').glob('*/budget_ledger.json')), 'Existing work without a manifest needs inspection.'
        for name, expected in EXPECTED.items():
            unpack_verified(INPUT/name, expected, BUNDLE)
        assert sha256(manifest) == SCIENTIFIC_SHA
        print('Staged original bundle; existing Drive results/recovery preserved.')
    CONTROL.mkdir(exist_ok=True)
    unpack_verified(INPUT/'msd_max_workers_control.zip', CONTROL_SHA, CONTROL)
print('No augmentation. Input data stays on the runtime; durable results stay in', RESULTS)


## 2. Check Drive headroom
Reserve at least 8 GiB for current recovery snapshots and metadata. Completed model checkpoints
are retired only after their evaluated results have been archived and verified.

Concurrency reserves 8 GiB of Drive headroom per active worker and 12 GiB of local SSD per worker.


In [ ]:
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
creds, _ = google.auth.default()
quota = build('drive', 'v3', credentials=creds, cache_discovery=False).about().get(fields='storageQuota').execute()['storageQuota']
if 'limit' in quota:
    free = int(quota['limit']) - int(quota['usage'])
    print(f'Drive quota headroom: {free/1024**3:.2f} GiB')
    assert free >= 8*1024**3, 'Free at least 8 GiB before training.'
else:
    print('Drive reports no fixed quota limit.')
assert shutil.disk_usage('/content').free >= 20*1024**3, 'At least 20 GiB runtime SSD headroom required.'


## 3. Create an isolated training environment
The notebook kernel is left intact. Package pins match the cluster recipe, including actual
dynamic-network-architectures 0.4.4. Installation is reused while this runtime survives.
Missing pip is bootstrapped from PyPA. Setup and campaign output is printed and saved locally.
CUDA initialization and numerical equivalence are checked in the next cell.


In [ ]:
VENV = Path('/content/hippo_msd_venv')
PYTHON = VENV/'bin/python'
def run_setup_logged(args, log_path):
    log_path = Path(log_path)
    tail = []
    with log_path.open('a', encoding='utf-8') as log:
        proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1, start_new_session=True)
        def drain():
            for line in proc.stdout:
                print(line, end='', flush=True)
                log.write(line); log.flush()
                tail.append(line)
                del tail[:-40]
        interrupted = False
        try:
            drain()
            result = proc.wait()
        except KeyboardInterrupt:
            interrupted = True
            print('Requesting a clean stop. Wait for the process and recovery upload to finish.')
            os.killpg(proc.pid, signal.SIGINT)
            drain()
            result = proc.wait()
        finally:
            proc.stdout.close()
        if result and not interrupted:
            raise RuntimeError(f'Command exited {result}. Full log: {log_path}\n' + ''.join(tail))

def ensure_training_pip(python, bootstrap, log_path):
    probe = subprocess.run([str(python), '-m', 'pip', '--version'], capture_output=True, text=True)
    if probe.returncode:
        import urllib.request
        urllib.request.urlretrieve('https://bootstrap.pypa.io/get-pip.py', str(bootstrap))
        run_setup_logged([str(python), str(bootstrap)], log_path)

if not PYTHON.exists():
    run_setup_logged([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(VENV)], '/content/hippo_msd_setup.log')
ensure_training_pip(PYTHON, Path('/content/hippo_msd_get_pip.py'), '/content/hippo_msd_setup.log')
run_setup_logged([str(PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check', '-r', str(BUNDLE/'requirements-colab.txt')], '/content/hippo_msd_setup.log')
def campaign(command, *extra):
    args = ([str(PYTHON), '-u', str(CONTROL/'scheduler.py'), command, '--bundle', str(BUNDLE), '--durable', str(RESULTS), *map(str, extra)]
            if command in ('run', 'report') else [str(PYTHON), '-u', str(BUNDLE/'runtime.py'), command, '--bundle', str(BUNDLE), '--durable', str(RESULTS), *map(str, extra)])
    run_setup_logged(args, Path('/content')/f'hippo_msd_{command}.log')
campaign('verify')
subprocess.run(['nvidia-smi'], check=True)


## 4. Required GPU preflight
Checks all 12 initializations.
Original CPU truncated-normal tensors are included for portability between Python builds;
the random draws and complete model hashes must still match the verified cluster recipe.
Then compares the original and accelerated code on this GPU: losses, gradients, optimizer/model
state, calibration, evaluation, RNG and checkpoint resume.
Its diagnostic updates do not count as scientific runs. A failure blocks training.
This can take several minutes; it is reused only with the same code, GPU and software.


In [ ]:
campaign('preflight')


## 5. Run or resume with maximum workers

`MAX_WORKERS = 0` chooses a ceiling from available CPU cores (two threads per experiment, with
two cores reserved for orchestration). That ceiling becomes the target immediately: for example,
a 12-core runtime targets five simultaneous workers. A positive value caps this target.
The scheduler fills eligible slots without waiting for solo memory measurements and refills
them after completed runs are archived. GPU activity and throughput are reported but do not
reduce the worker target. A full target is not a guarantee of maximum completed runs per hour.

Unmeasured model/condition pairs receive provisional allowances of at least 12 GiB GPU memory
and 8 GiB system RAM per worker; known same-model peaks and live observations can raise them.
Measured pairs reuse full-run profiles with margins. Every admission accounts for other workers'
reserved future growth plus GPU headroom of at least 8 GiB/15% and 4 GiB system RAM.
These allowances are estimates, not proven peak bounds. Out-of-memory or other worker failure
stops new launches and saves other workers; failed charged work is preserved for an audit.

Interrupted tasks have launch priority, with other eligible work filling spare slots.
Fresh baselines have priority over new constraints; each new constraint waits for its own matched
baseline archive. Other folds/seeds can continue concurrently while a baseline finishes.
Each task still has its own original seeds, batch size 1, BF16, calibration, losses, optimizer,
update schedule, checkpoints and 31,976,450,667,090 optimization-FLOP cap. No augmentation.
Training can remain partly idle when Drive is the bottleneck; do not equate VRAM occupancy with speed.

New run checkpoints and recovery tar staging use `/mnt/local-scratch` when it is mounted separately.
The `STORAGE` line prints the actual location. Existing interrupted directories remain in place;
all 504 task IDs and their archive/recovery paths stay unchanged. The normal recovery snapshot is
still verified on Drive, and every update keeps its original durable accounting write/readback.
Storage headroom is checked on the actual destination. Do not manually delete work-folder symlinks.
`WAIT` messages explain any unfilled slots: memory, storage or matched-baseline dependencies.
Startup verification, archive I/O, stopping and the final few tasks can also leave slots unfilled.

Before training, `STARTUP` messages show the verification phase, checked/total/verified folder counts,
current archive and filename. Each finished folder is reported. Every 30 seconds a heartbeat shows
`PROGRESS` if work advanced or `STILL WAITING` otherwise, with time since the last progress event.
The GPU may remain unused until verification finishes; a blocked Drive read cannot be bypassed.
The same status is saved locally at `/content/hippo_msd_bundle/parallel_status/startup.json`.
Run/recovery directories are listed once to avoid probing hundreds of absent Drive folders.
Each archive file is copied once to a temporary local snapshot; the unchanged verifier checks all
hashes, byte counts, readability, task identity, source binding, acknowledgment and cleanup locally.
Metrics and memory profiles use the same verified bytes. The local snapshot is removed immediately
after use; Drive files are never modified by verification. Every restart checks fresh Drive bytes.

During training, every minute the notebook prints completed count, active workers, GPU activity, memory and charged
update counts. Ledger-write/readback time, checkpoint time and memory peaks are archived per run.
Existing completed archives are never rewritten. No extra scientific runs or tuning trials are added.

**To stop:** interrupt once and wait for every worker's recovery upload. A worker failure stops new
launches and asks the others to save. Hard VM loss still requires an audit if charged updates are
newer than durable checkpoints; accounting is never reset to hide lost work.

`SESSION_HOURS` counts from starting this cell. Set it below your **remaining** runtime allowance,
including time already spent in the previous notebook. The default is 8 more hours for this handover;
on a newly allocated Pro+ runtime you may use 22. Colab can terminate earlier. Existing STOP files are respected.


In [ ]:
MAX_NEW_RUNS = 504  # Upper bound on remaining tasks launched this session, not extra repeats.
MAX_WORKERS = 0    # Fill the CPU-supported maximum immediately; no solo profiling.
SESSION_HOURS = 8.0  # Remaining time for this session; fresh runtime may use 22.0.
campaign('run', '--max-runs', MAX_NEW_RUNS, '--hours', SESSION_HOURS, '--workers', MAX_WORKERS)


## 6. Results so far
Each gain is paired with the same model/fold/seed baseline. A final mean is shown only after all
nine cells of that model/condition are present: average the three seeds within each fold, then
average the three fold means. The displayed SD is across fold means, not nine independent subjects.
Architecture remains confounded with baseline loss/optimizer; this is validation, not an independent test set.


In [ ]:
campaign('report')
result = json.loads((RESULTS/'comparison.json').read_text())
import pandas as pd
print(f"Complete: {result['completed']}/{result['total']}")
display(pd.DataFrame(result['aggregate']))
display(pd.DataFrame(result['runs']))


## 7. Curves and GPU use
Per-run `metrics.csv`, `performance.json`, `calibration.json`, full diagnostics, configuration and
integer FN/FP/A–P swaps live under `results_noaug_v1/runs/`. Inspect training and validation together.
Use the runtime's GPU utilization display or `nvidia-smi`; high utilization alone does not establish
a faster or scientifically equivalent experiment. Do not raise batch size or change precision.

Scheduler provenance, preservation inventory and minute-by-minute timing are under `results_noaug_v1/parallel/`.


In [ ]:
import matplotlib.pyplot as plt
finished = result['runs']
if finished:
    chosen = finished[-1]['id']
    curve = pd.read_csv(RESULTS/'runs'/chosen/'metrics.csv')
    plt.plot(curve['compute_fraction'], 100*curve['macro_hard_dice'], marker='.')
    plt.xlabel('Fraction of optimization FLOP budget')
    plt.ylabel('Validation Dice (%)')
    plt.title(chosen)
    plt.grid(alpha=.25)
    plt.show()
subprocess.run(['nvidia-smi', '--query-gpu=name,utilization.gpu,memory.used,memory.total', '--format=csv'], check=True)


## Resume later
Open this same notebook, use the same GPU type, and rerun setup/preflight/run cells. Keep the same
results folder and uploaded archives. No laptop collector or cluster scheduler is involved.
Do not run two copies against the same output directory. A changed GPU/software environment is
rejected instead of mixing its results into the nine-cell comparisons.

Google Drive's quota API is documented [here](https://developers.google.com/workspace/drive/api/reference/rest/v3/about).
Colab resources and background execution remain subject to the [Colab FAQ](https://research.google.com/colaboratory/faq.html).
Disconnect and delete the GPU runtime when finished; the verified Drive results remain.
